In [8]:


import folium
import pandas as pd
import numpy as np
from folium.plugins import HeatMap, MousePosition
import branca.colormap as cm

def create_interactive_map(cv_predictions, output_html="pune_no2_interactive.html"):
    """
    Creates a zoomable, pannable NO2 map with:
    - Real Google/OpenStreetMap tiles underneath
    - Colour-coded NO2 circles for each pixel
    - Click any point → popup shows NO2 value, date, lat/lon
    - Layer toggle: annual mean / seasonal / exceedance
    - Ground station markers with actual measured values
    """

    df = pd.read_csv(cv_predictions)

    # Annual mean per pixel
    annual = df.groupby(["latitude","longitude"],
                         as_index=False)["NO2_pred_ugm3"].mean()

    # ── Base map — centred on Pune ─────────────────────────────
    m = folium.Map(
        location=[18.52, 73.86],
        zoom_start=12,
        tiles=None               # we add tiles manually below
    )

    # ── Map tile layers (user can switch between them) ─────────
    folium.TileLayer(
        "OpenStreetMap", name="Street map", control=True
    ).add_to(m)

    folium.TileLayer(
        tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
        attr="Google",
        name="Satellite (Google)",
        control=True
    ).add_to(m)

    folium.TileLayer(
        tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
        attr="Google",
        name="Google Maps",
        control=True
    ).add_to(m)

    # ── Colour scale — green to red ────────────────────────────
    colormap = cm.LinearColormap(
        colors=["#00E400","#FFFF00","#FF7E00","#FF0000","#8F3F97"],
        vmin=0, vmax=150,
        caption="Predicted NO2 (µg/m³)"
    )
    colormap.add_to(m)

    # ── NO2 circle markers ─────────────────────────────────────
    no2_layer = folium.FeatureGroup(name="NO2 — Annual mean", show=True)

    for _, row in annual.iterrows():
        no2_val  = row["NO2_pred_ugm3"]
        colour   = colormap(no2_val)

        # WHO/NAAQS status label
        if no2_val > 80:
            status = "Exceeds NAAQS limit"
        elif no2_val > 25:
            status = "Exceeds WHO guideline"
        else:
            status = "Within WHO guideline"

        popup_html = f"""
        <div style='font-family:Arial;font-size:13px;min-width:180px'>
          <b style='font-size:15px'>NO2: {no2_val:.1f} µg/m³</b><br>
          <span style='color:{"red" if no2_val>80 else "orange" if no2_val>25 else "green"}'>
            {status}
          </span><br><br>
          <b>Location:</b> {row['latitude']:.4f}°N, {row['longitude']:.4f}°E<br>
          <b>WHO limit:</b>  25 µg/m³<br>
          <b>NAAQS limit:</b> 80 µg/m³<br>
        </div>
        """

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=8,
            color=colour,
            fill=True,
            fill_color=colour,
            fill_opacity=0.75,
            weight=0.5,
            popup=folium.Popup(popup_html, max_width=220),
            tooltip=f"NO2: {no2_val:.1f} µg/m³"
        ).add_to(no2_layer)

    no2_layer.add_to(m)

    # ── Ground station markers ─────────────────────────────────
    STATIONS = {
        "Gavalinagar":   (18.63673, 73.82487),
        "Katraj Dairy":  (18.45445, 73.85416),
        "Park Wakad":    (18.59700, 73.76200),
        "SPPU":          (18.52900, 73.85100),
        "Thergaon":      (18.61100, 73.78400),
        "Bhumkar Nagar": (18.59800, 73.77300),
        "Hadapsar":      (18.50300, 73.93900),
        "Panchawati":    (18.53600, 73.82600),
        "Savta Mali":    (18.62800, 73.80600),
        "Nigdi":         (18.65400, 73.78800),
    }

    station_layer = folium.FeatureGroup(name="Ground stations", show=True)
    for name, (lat, lon) in STATIONS.items():
        folium.Marker(
            location=[lat, lon],
            icon=folium.Icon(color="black", icon="tachometer", prefix="fa"),
            popup=folium.Popup(
                f"<b>{name}</b><br>CPCB/MPCB monitoring station<br>"
                f"Lat: {lat}  Lon: {lon}", max_width=200),
            tooltip=f"Station: {name}"
        ).add_to(station_layer)
    station_layer.add_to(m)

    # ── Mouse position display (shows lat/lon as you hover) ────
    MousePosition(
        position="bottomleft",
        separator=" | ",
        prefix="Lat/Lon:",
        num_digits=5
    ).add_to(m)

    # ── Layer control (toggle layers on/off) ───────────────────
    folium.LayerControl(collapsed=False).add_to(m)

    # ── Save ───────────────────────────────────────────────────
    m.save(output_html)
    print(f"Interactive map saved → {output_html}")
    print("Open this file in any browser — no server needed")

# Run it
create_interactive_map("model_outputs/cv_predictions.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'model_outputs/cv_predictions.csv'